<a href="https://colab.research.google.com/github/Isab-os/colab/blob/main/limpeza_satisfacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# dc_satisfacao
---
#### 1° prompt para diagnóstico:
Anexo o arquivo dc_satisfacao. Gere um código de diagnóstico que reporte, por coluna: tipo inferido, percentual de nulos, número de valores distintos, os 5 mais frequentes e possíveis problemas de padronização. Não altere nada ainda, só me mostre o retrato da base.

In [17]:
import pandas as pd

# Load the dataset
file_path = '/content/sample_data/dc_satisfacao (1).csv'
df_satisfacao = pd.read_csv(file_path)

print(f"Dataset '{file_path}' loaded successfully. Shape: {df_satisfacao.shape}")

Dataset '/content/sample_data/dc_satisfacao (1).csv' loaded successfully. Shape: (103, 5)


## Diagnóstico dos dados:

In [18]:
diagnosis_report = {}

for col in df_satisfacao.columns:
    col_data = df_satisfacao[col]

    # Inferred Type
    inferred_type = pd.api.types.infer_dtype(col_data)

    # Percentage of null values
    null_percentage = col_data.isnull().sum() / len(col_data) * 100

    # Number of distinct values
    num_distinct = col_data.nunique()

    # Top 5 most frequent values
    top_5_frequent = col_data.value_counts(dropna=False).head(5).to_dict()

    # Possible standardization issues (basic checks)
    standardization_issues = []
    if inferred_type == 'mixed-integer-float':
        standardization_issues.append('Mixed integer and float types.')
    elif inferred_type == 'mixed':
        standardization_issues.append('Mixed types detected (e.g., numbers and strings).')
    elif inferred_type == 'string' and num_distinct > 1 and len(str(col_data.mode().iloc[0])) > 20 and num_distinct < len(col_data) / 2: # Heuristic for potentially verbose strings that might need cleaning
        standardization_issues.append('Potentially verbose string values; consider cleaning/standardizing.')
    elif 'object' in str(col_data.dtype) and inferred_type not in ['string', 'datetime', 'boolean'] and num_distinct > 1: # check for non-string object columns
        # Try to convert to numeric to see if there are non-numeric values
        try:
            pd.to_numeric(col_data.dropna(), errors='raise')
        except ValueError:
            standardization_issues.append('Object type column contains non-numeric values that could be standardized.')

    diagnosis_report[col] = {
        'inferred_type': inferred_type,
        'null_percentage': f"{null_percentage:.2f}%",
        'num_distinct_values': num_distinct,
        'top_5_frequent_values': top_5_frequent,
        'potential_standardization_issues': standardization_issues
    }

# Display the report
for col, info in diagnosis_report.items():
    print(f"\n--- Column: {col} ---")
    for key, value in info.items():
        print(f"  {key.replace('_', ' ').capitalize()}: {value}")

df_satisfacao.head()


--- Column: pesquisa_id ---
  Inferred type: string
  Null percentage: 0.00%
  Num distinct values: 103
  Top 5 frequent values: {'PSQ0001': 1, 'PSQ0002': 1, 'PSQ0003': 1, 'PSQ0004': 1, 'PSQ0005': 1}
  Potential standardization issues: []

--- Column: projeto_id ---
  Inferred type: string
  Null percentage: 0.00%
  Num distinct values: 103
  Top 5 frequent values: {'PRJ0001': 1, 'PRJ0003': 1, 'PRJ0004': 1, 'PRJ0005': 1, 'PRJ0006': 1}
  Potential standardization issues: []

--- Column: data_pesquisa ---
  Inferred type: string
  Null percentage: 0.00%
  Num distinct values: 100
  Top 5 frequent values: {'2024-11-25': 2, '2025-12-22': 2, '2026-06-22': 2, '2024-05-27': 1, '2025-03-27': 1}
  Potential standardization issues: []

--- Column: nota_nps ---
  Inferred type: floating
  Null percentage: 7.77%
  Num distinct values: 8
  Top 5 frequent values: {10.0: 31, 9.0: 22, 8.0: 19, 7.0: 11, nan: 8}
  Potential standardization issues: []

--- Column: comentario ---
  Inferred type: string


,pesquisa_id,projeto_id,data_pesquisa,nota_nps,comentario
0,PSQ0001,PRJ0001,2024-05-27,NaN,"Resultado ok, prazo apertado."
1,PSQ0002,PRJ0003,2025-10-20,10.0,Documentação impecável.
2,PSQ0003,PRJ0004,2024-05-08,9.0,Documentação impecável.
3,PSQ0004,PRJ0005,2025-03-27,7.0,"Bom projeto, comunicação pode melhorar."
4,PSQ0005,PRJ0006,17/07/2025,6.0,Suporte demorou a responder.


# Plano de limpeza
## Eu mando
- "Com base no diagnóstico acima, proponha um plano de limpeza e padronização em etapas, com o motivo de cada uma e o impacto esperado em número de linhas. Antes de escrever o código, me mostre o plano para eu aprovar."
---
##IA planeja:
- Ajustar formatos das datas para 'dd/mm/yyyy' (string): Normalizar todos os formatos de data reconhecidos na coluna 'data_pesquisa' para o formato de string 'dd/mm/yyyy'. Valores que não puderem ser interpretados ou já estiverem neste formato devem ser mantidos como estão. Esta etapa também fornecerá um resumo das conversões realizadas.

- Converter 'data_pesquisa' para tipo Datetime: Após a padronização para o formato de string 'dd/mm/yyyy', converter a coluna 'data_pesquisa' para o tipo de dado datetime. Valores que não forem válidos para conversão para datetime se tornarão NaT (Not a Time).

- Tratar Valores Ausentes e Tipo em 'nota_nps': Preencher os valores ausentes (NaN) na coluna 'nota_nps' com nulos (NaN, compatível com tipo de dados de inteiro com nulos) e converter a coluna para o tipo inteiro (int), garantindo que os valores estejam no intervalo de 0 a 10. Não haverá impacto no número de linhas (manterá 103 linhas).

- Final Task: Apresentar um resumo das ações de limpeza e padronização realizadas e o estado final do DataFrame, confirmando que as colunas foram tratadas conforme o plano.

# Task
O usuário quer realizar a limpeza e padronização de dados no DataFrame df_satisfacao. O plano inclui normalizar os formatos de data, converter datas para objetos datetime, lidar com valores ausentes em 'nota_nps' e convertê-los para tipo inteiro, e por fim, resumir as ações de limpeza e o estado final do DataFrame.

## Ajustar formatos das datas para 'yyyy/mm/dd' (string)

### Subtask:
Normalizar todos os formatos de data reconhecidos na coluna 'data_pesquisa' para o formato de string 'yyyy/mm/dd'.


**Raciocinio da IA**:
Eu preciso implementar a função para normalizar os formatos de data e aplicá-la na coluna 'data_pesquisa', conforme as instruções. Esse passo vai incluir identificar os formatos, criar a função, aplicá-la e contar as conversões.



In [21]:
import pandas as pd

# --- 1. Dicionário de meses abreviados em português ---
meses_pt = {
    "jan": "Jan", "fev": "Feb", "mar": "Mar", "abr": "Apr", "mai": "May", "jun": "Jun",
    "jul": "Jul", "ago": "Aug", "set": "Sep", "out": "Oct", "nov": "Nov", "dez": "Dec"
}

# --- 2. Lista de formatos ---
date_formats = [
    '%Y-%m-%d',      # YYYY-MM-DD
    '%d/%m/%Y',      # DD/MM/YYYY
    '%m/%d/%Y',      # MM/DD/YYYY
    '%Y/%m/%d',      # YYYY/MM/DD
    '%d-%m-%Y',      # DD-MM-YYYY
    '%d-%b-%Y',      # DD-Mon-YYYY (abreviado em inglês)
]

# --- 3. Carregar dataset ---
try:
    df_satisfacao = pd.read_csv('/content/sample_data/dc_satisfacao (1).csv')
    print("Dataset carregado com sucesso.")
except FileNotFoundError:
    print("Erro: arquivo não encontrado.")
    exit()

# --- 4. Função para padronizar datas ---
def format_date(date_str):
    if pd.isna(date_str):
        return date_str
    date_str = str(date_str).strip().lower()

    # Substituir abreviações em português por inglês
    for pt, en in meses_pt.items():
        if pt in date_str:
            date_str = date_str.replace(pt, en)

    # Testar formatos conhecidos
    for fmt in date_formats:
        try:
            parsed_date = pd.to_datetime(date_str, format=fmt, errors='raise')
            # Forçar saída no formato yyyy/mm/dd
            return parsed_date.strftime('%Y/%m/%d')
        except ValueError:
            continue

    # Se não conseguir converter, retorna original
    return date_str

# --- 5. Aplicar padronização ---
df_satisfacao['data_pesquisa'] = df_satisfacao['data_pesquisa'].apply(format_date)

print("Datas padronizadas para yyyy/mm/dd (formato ano/mês/dia):")
print(df_satisfacao[['pesquisa_id','data_pesquisa']].head(10))


Dataset carregado com sucesso.
Datas padronizadas para yyyy/mm/dd (formato ano/mês/dia):
  pesquisa_id data_pesquisa
0     PSQ0001    2024/05/27
1     PSQ0002    2025/10/20
2     PSQ0003    2024/05/08
3     PSQ0004    2025/03/27
4     PSQ0005    2025/07/17
5     PSQ0006    2024/11/06
6     PSQ0007    2025/09/15
7     PSQ0008    2025/01/21
8     PSQ0009    2024/11/27
9     PSQ0010    2025/12/29


datas que fica ambiguas (antes da formatação) devido ao formato não ser especificado devidamente (podem ser tanto 'dd/mm/yyyy' ou 'mm/dd/yyyy')
Linha 5 → 17/07/2025 → pode ser 17 de julho ou 7 de novembro (se fosse mm/dd).

Linha 12 → 26/03/2025 → pode ser 26 de março ou 3 de junho.

Linha 19 → 30/04/2025 → pode ser 30 de abril ou 4 de outubro.

Linha 21 → 11/06/2024 → pode ser 11 de junho ou 6 de novembro.

Linha 26 → 21/02/2024 → pode ser 21 de fevereiro ou 2 de dezembro.

Linha 33 → 16/06/2025 → pode ser 16 de junho ou 6 de novembro.

Linha 35 → 04/08/2024 → pode ser 4 de agosto ou 8 de abril.

Linha 40 → 15/05/2026 → pode ser 15 de maio ou 5 de novembro.

Linha 42 → 11/04/2024 → pode ser 11 de abril ou 4 de novembro.

Linha 47 → 03/06/2026 → pode ser 3 de junho ou 6 de março.

Linha 54 → 11/03/2025 → pode ser 11 de março ou 3 de novembro.

Linha 61 → 10/11/2025 → pode ser 10 de novembro ou 11 de outubro.

Linha 63 → 12/09/2024 → pode ser 12 de setembro ou 9 de dezembro.

Linha 68 → 02/10/2024 → pode ser 2 de outubro ou 10 de fevereiro.

Linha 71 → 05/09/2024 → pode ser 5 de setembro ou 9 de maio.

Linha 72 → 02/12/2024 → pode ser 2 de dezembro ou 12 de fevereiro.

Linha 82 → 05/05/2025 → pode ser 5 de maio ou 5 de maio (nesse caso é simétrico, mas ainda ambíguo).

Linha 89 → 25/08/2025 → pode ser 25 de agosto ou 8 de maio.

Linha 96 → 18/06/2026 → pode ser 18 de junho ou 6 de agosto.

Linha 103 → 13/02/2026 → pode ser 13 de fevereiro ou 2 de dezembro.


## Converter 'data_pesquisa' para tipo Datetime

### Subtask:
Converta a coluna 'data_pesquisa' para o tipo de dado datetime. Valores que não forem datas válidas vão virar NaT (Not a Time).


**Raciocínio da IA**: Eu preciso converter a coluna 'data_pesquisa' para o tipo de dado datetime conforme especificado nas instruções. Usar pd.to_datetime() com errors='coerce' vai lidar com qualquer data que não possa ser interpretada, convertendo-a em NaT.


In [20]:
df_satisfacao['data_pesquisa'] = pd.to_datetime(df_satisfacao['data_pesquisa'], format='%d/%m/%Y', errors='coerce')

print("Data types after converting 'data_pesquisa' to datetime:")
print(df_satisfacao.dtypes)

print("\nFirst 5 rows of the DataFrame with updated 'data_pesquisa':")
print(df_satisfacao.head())

Data types after converting 'data_pesquisa' to datetime:
pesquisa_id              object
projeto_id               object
data_pesquisa    datetime64[ns]
nota_nps                float64
comentario               object
dtype: object

First 5 rows of the DataFrame with updated 'data_pesquisa':
  pesquisa_id projeto_id data_pesquisa  nota_nps  \
0     PSQ0001    PRJ0001    2024-05-27       NaN   
1     PSQ0002    PRJ0003    2025-10-20      10.0   
2     PSQ0003    PRJ0004    2024-05-08       9.0   
3     PSQ0004    PRJ0005    2025-03-27       7.0   
4     PSQ0005    PRJ0006    2025-07-17       6.0   

                                comentario  
0            Resultado ok, prazo apertado.  
1                  Documentação impecável.  
2                  Documentação impecável.  
3  Bom projeto, comunicação pode melhorar.  
4             Suporte demorou a responder.  


**Raciocínio da IA**: O passo anterior converteu com sucesso a coluna 'data_pesquisa' para datetime. Agora, eu preciso confirmar se errors='coerce' funcionou como esperado, verificando e imprimindo a contagem de valores NaT na coluna 'data_pesquisa', além de imprimir o número de valores não nulos.



In [10]:
df_satisfacao['data_pesquisa'] = pd.to_datetime(df_satisfacao['data_pesquisa'], format='%d/%m/%Y', errors='coerce')

print("Data types after converting 'data_pesquisa' to datetime:")
print(df_satisfacao.dtypes)

print("\nFirst 5 rows of the DataFrame with updated 'data_pesquisa':")
print(df_satisfacao.head())

# Check for NaT values (values that couldn't be parsed)
nat_count = df_satisfacao['data_pesquisa'].isna().sum()
print(f"\nNumber of NaT values in 'data_pesquisa': {nat_count}")
print(f"Number of non-null values in 'data_pesquisa': {df_satisfacao['data_pesquisa'].count()}")

Data types after converting 'data_pesquisa' to datetime:
pesquisa_id              object
projeto_id               object
data_pesquisa    datetime64[ns]
nota_nps                float64
comentario               object
dtype: object

First 5 rows of the DataFrame with updated 'data_pesquisa':
  pesquisa_id projeto_id data_pesquisa  nota_nps  \
0     PSQ0001    PRJ0001    2024-05-27       NaN   
1     PSQ0002    PRJ0003    2025-10-20      10.0   
2     PSQ0003    PRJ0004    2024-05-08       9.0   
3     PSQ0004    PRJ0005    2025-03-27       7.0   
4     PSQ0005    PRJ0006    2025-07-17       6.0   

                                comentario  
0            Resultado ok, prazo apertado.  
1                  Documentação impecável.  
2                  Documentação impecável.  
3  Bom projeto, comunicação pode melhorar.  
4             Suporte demorou a responder.  

Number of NaT values in 'data_pesquisa': 0
Number of non-null values in 'data_pesquisa': 103


## Tratar Valores Ausentes e Tipo em 'nota_nps'

### Subtask:
Preencher os valores ausentes (NaN) na coluna 'nota_nps' com nulos (NaN, compatível com tipo de dados de inteiro com nulos) e converter a coluna para o tipo inteiro (int), garantindo que os valores estejam no intervalo de 0 a 10. Não haverá impacto no número de linhas (manterá 103 linhas).


**Reasoning**:
Convert 'nota_nps' to a numeric type first, coercing errors to NaN, then convert it to the nullable integer type 'Int64'.



In [11]:
df_satisfacao['nota_nps'] = pd.to_numeric(df_satisfacao['nota_nps'], errors='coerce')
df_satisfacao['nota_nps'] = df_satisfacao['nota_nps'].astype('Int64')

print("Data types after converting 'nota_nps':")
print(df_satisfacao.dtypes)

print(f"\nNumber of null values in 'nota_nps': {df_satisfacao['nota_nps'].isnull().sum()}")
print("\nFirst 5 rows of the DataFrame with updated 'nota_nps':")
print(df_satisfacao.head())

Data types after converting 'nota_nps':
pesquisa_id              object
projeto_id               object
data_pesquisa    datetime64[ns]
nota_nps                  Int64
comentario               object
dtype: object

Number of null values in 'nota_nps': 8

First 5 rows of the DataFrame with updated 'nota_nps':
  pesquisa_id projeto_id data_pesquisa  nota_nps  \
0     PSQ0001    PRJ0001    2024-05-27      <NA>   
1     PSQ0002    PRJ0003    2025-10-20        10   
2     PSQ0003    PRJ0004    2024-05-08         9   
3     PSQ0004    PRJ0005    2025-03-27         7   
4     PSQ0005    PRJ0006    2025-07-17         6   

                                comentario  
0            Resultado ok, prazo apertado.  
1                  Documentação impecável.  
2                  Documentação impecável.  
3  Bom projeto, comunicação pode melhorar.  
4             Suporte demorou a responder.  


Verifica a faixa de valores da coluna 'nota_nps' e confirma o a contagem de linhas.



In [12]:
print(f"Minimum value in 'nota_nps': {df_satisfacao['nota_nps'].min()}")
print(f"Maximum value in 'nota_nps': {df_satisfacao['nota_nps'].max()}")
print(f"Total number of rows in DataFrame: {len(df_satisfacao)}")

Minimum value in 'nota_nps': 3
Maximum value in 'nota_nps': 10
Total number of rows in DataFrame: 103


## Save DataFrame to CSV


**Reasoning**:
Save the DataFrame `df_satisfacao` to a new CSV file named 'dc_satisfacao_limpo.csv' without including the index.



In [22]:
df_satisfacao.to_csv('dc_satisfacao_limpo.csv', index=False)
print("DataFrame 'df_satisfacao' salvo como 'dc_satisfacao_limpo.csv' com sucesso.")

DataFrame 'df_satisfacao' salvo como 'dc_satisfacao_limpo.csv' com sucesso.
